# 从示范到生成策略：岔路口冒烟实验

对应教程 7.8。机器人在底部中央出发，正中间有一堵墙（圆形障碍），左上和右上各有一个目标——
走哪边都对，专家一半时间向左绕、一半时间向右绕。
我们依次训练三档策略（MSE 回归、离散分类、迷你扩散），看谁会"撞墙"、谁会"选边"。全程 CPU。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from hwm.robot import (
    FORK_START, FORK_TARGETS, FORK_OBSTACLE, make_fork_dataset, fork_success_rate,
)
torch.manual_seed(0)

## 1. 双模态示范数据

起点附近，同一个状态对应两组方向不同的动作：向左绕（模式 0）或向右绕（模式 1）。
两组动作的平均值直指正中央——也就是直指那堵墙。数据同时提供逐步动作和 6 步动作序列（chunk）。

In [2]:
data = make_fork_dataset(num_episodes=128, horizon=20, chunk_size=6, seed=0)
for name, value in data.items(): print(f'{name:14s}', tuple(value.shape))
start = torch.from_numpy(FORK_START)[None]
near = (data['states'] - start).abs() < 0.08
near = near.all(dim=-1)
print('起点附近 模式0 的平均动作（向左绕）:', data['actions'][near & (data['modes'] == 0)].mean(0).round(decimals=2).tolist())
print('起点附近 模式1 的平均动作（向右绕）:', data['actions'][near & (data['modes'] == 1)].mean(0).round(decimals=2).tolist())
print('起点附近 全部动作的平均（直指墙）  :', data['actions'][near].mean(0).round(decimals=2).tolist())
assert data['states'].shape == data['actions'].shape == (128 * 20, 2)
assert data['action_chunks'].shape[1:] == (6, 2)

states         (2560, 2)
actions        (2560, 2)
modes          (2560,)
chunk_states   (1920, 2)
action_chunks  (1920, 6, 2)
起点附近 模式0 的平均动作（向左绕）: [-0.4000000059604645, 0.9200000166893005]
起点附近 模式1 的平均动作（向右绕）: [0.4000000059604645, 0.9200000166893005]
起点附近 全部动作的平均（直指墙）  : [0.009999999776482582, 0.9200000166893005]


## 2. 第一档：MSE 回归

最小均方误差的最优解是条件均值——向左绕和向右绕一平均，剩下"直走"。
直走会撞上正中央的墙：环境会挡住进入墙内的移动。有的轨迹卡死在墙下，有的沿墙滑出去侥幸成功——
所以除了成功率，必须同时看撞墙次数（这正是 7.9 要求"成功率与碰撞率同时报告"的原因）。

In [3]:
mse_policy = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2), nn.Tanh(),
)
optimizer = torch.optim.Adam(mse_policy.parameters(), lr=3e-3)
for epoch in range(500):
    optimizer.zero_grad()
    loss = F.mse_loss(mse_policy(data['states']), data['actions'])
    loss.backward()
    optimizer.step()
print('final mse loss:', round(float(loss.detach()), 4))
print('策略在起点的输出（应接近直走）:', mse_policy(start)[0].detach().round(decimals=2).tolist())

final mse loss: 0.0947
策略在起点的输出（应接近直走）: [0.019999999552965164, 1.0]


In [4]:
mse_result = fork_success_rate(
    lambda: (lambda pos: mse_policy(torch.from_numpy(pos)[None])[0].detach().numpy()), seed=1,
)
print('MSE 闭环成功率:', mse_result['success_rate'], ' 平均撞墙次数:', round(mse_result['mean_collisions'], 1))
print('一条卡死的轨迹:', mse_result['runs'][0]['trajectory'][::4].round(2).tolist())

MSE 闭环成功率: 0.65625  平均撞墙次数: 7.6
一条卡死的轨迹: [[0.49000000953674316, 0.10999999940395355], [0.4699999988079071, 0.3499999940395355], [0.4699999988079071, 0.3499999940395355], [0.4699999988079071, 0.3499999940395355], [0.4699999988079071, 0.3499999940395355], [0.4699999988079071, 0.3499999940395355], [0.4699999988079071, 0.3499999940395355]]


## 3. 第二档：离散化 + 承诺（RT-1 / VQ-BeT 的做法）

不再回归动作，而是分类"这条轨迹走哪边"（左/右两类），**每条轨迹开头采样一次，然后承诺执行**。
分类不会平均：起点附近两类概率各约一半，采样就是选边。真 ACT 的 CVAE 做的是同一件事——把"求均值"换成"选模式"。
选中模式后用一个简单的跟踪器朝该侧目标走（含绕墙），跟踪器是"执行器"，不是学习重点。

In [5]:
mode_clf = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = torch.optim.Adam(mode_clf.parameters(), lr=3e-3)
for epoch in range(500):
    optimizer.zero_grad()
    loss = F.cross_entropy(mode_clf(data['states']), data['modes'])
    loss.backward()
    optimizer.step()
print('final ce loss:', round(float(loss.detach()), 4))

def track(pos, mode):
    direction = FORK_TARGETS[mode] - pos
    direction = direction / (np.linalg.norm(direction) + 1e-6)
    to_wall = FORK_OBSTACLE - pos
    if np.linalg.norm(to_wall) < 0.3 and float(direction @ to_wall) > 0:
        side = np.array([-direction[1], direction[0]], dtype=np.float32)
        if mode == 1:
            side = -side
        direction = 0.35 * direction + 0.65 * side
        direction = direction / (np.linalg.norm(direction) + 1e-6)
    return direction

def make_cls_policy():
    mode = {'value': None}  # 每条轨迹重置：第一次调用时采样模式，之后承诺跟踪
    def policy(pos):
        if mode['value'] is None:
            logits = mode_clf(torch.from_numpy(pos)[None])
            mode['value'] = int(torch.distributions.Categorical(logits=logits).sample())
        return track(pos, mode['value'])
    return policy

cls_result = fork_success_rate(make_cls_policy, seed=1)
print('离散分类+承诺 闭环成功率:', cls_result['success_rate'])

final ce loss: 0.0419
离散分类+承诺 闭环成功率: 1.0


## 4. 第三档：迷你扩散策略，一次生成整段动作序列

扩散策略学会把噪声逐步擦成"像专家的动作序列"——注意它一次预测的是 **6 步序列**，不是单步动作。
起点附近它学到的是两簇序列（向左绕 / 向右绕），采样一簇就是选边，而且序列里带着绕墙的弧线。
这里用 100 步 DDPM，CPU 约一分钟训完。

In [6]:
T = 100
betas = torch.linspace(1e-3, 0.1, T)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)
CHUNK = 6

eps_net = nn.Sequential(
    nn.Linear(CHUNK * 2 + 3, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, CHUNK * 2),
)
optimizer = torch.optim.Adam(eps_net.parameters(), lr=1e-3)
for step in range(6000):
    index = torch.randint(0, len(data['chunk_states']), (256,))
    a0 = data['action_chunks'][index].reshape(256, -1)
    x = data['chunk_states'][index]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(a0)
    a_t = abar[t].sqrt()[:, None] * a0 + (1 - abar[t]).sqrt()[:, None] * eps
    pred = eps_net(torch.cat((a_t, x, t[:, None].float() / T), dim=-1))
    loss = F.mse_loss(pred, eps)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print('final diffusion loss:', round(float(loss.detach()), 4))

final diffusion loss: 0.1323


In [7]:
@torch.no_grad()
def sample_chunk(pos):
    a = torch.randn(CHUNK * 2)
    x = torch.from_numpy(pos).float()
    for t in reversed(range(T)):
        eps = eps_net(torch.cat((a, x, torch.tensor([t / T])))[None])[0]
        a = (a - (1 - alphas[t]) / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            a = a + betas[t].sqrt() * torch.randn(CHUNK * 2)
    return a.clamp(-1, 1).reshape(CHUNK, 2).numpy()

def make_chunk_policy():
    state = {'chunk': None, 'index': 0}  # 采样一段序列，开环执行完再重采样
    def policy(pos):
        if state['index'] == 0:
            state['chunk'] = sample_chunk(pos)
        action = state['chunk'][state['index']]
        state['index'] = (state['index'] + 1) % CHUNK
        return action
    return policy

dif_result = fork_success_rate(make_chunk_policy, seed=1)
print('扩散+分块执行 闭环成功率:', dif_result['success_rate'],
    ' 平均撞墙次数:', round(dif_result['mean_collisions'], 1))
first_steps = np.stack([sample_chunk(FORK_START)[0] for _ in range(8)])
print('起点处 8 次采样的第一步（应看到多个方向，而不是都指向墙）:', first_steps.round(2).tolist())

扩散+分块执行 闭环成功率: 0.90625  平均撞墙次数: 0.0
起点处 8 次采样的第一步（应看到多个方向，而不是都指向墙）: [[0.4399999976158142, 1.0], [0.18000000715255737, 0.9700000286102295], [0.4300000071525574, 0.9300000071525574], [1.0, 0.23999999463558197], [0.38999998569488525, 0.9200000166893005], [0.4000000059604645, 0.8399999737739563], [-1.0, 0.4300000071525574], [0.3799999952316284, 0.8999999761581421]]


## 5. 对照实验：每步重新规划会怎样

同一个扩散模型，改成每步都重新采样一整段序列、只执行第一步（replan）。
分布还是那两簇，但在岔口附近每步采到的序列方向不一致，第一步左右横跳、平均指向墙——
撞墙次数应明显高于分块执行。这就是"会采样"和"会执行"的区别，也是 action chunking 存在的理由。

In [8]:
replan_result = fork_success_rate(lambda: (lambda pos: sample_chunk(pos)[0]), seed=1)
print('扩散+每步重规划 闭环成功率:', replan_result['success_rate'],
    ' 平均撞墙次数:', round(replan_result['mean_collisions'], 1))

扩散+每步重规划 闭环成功率: 0.4375  平均撞墙次数: 2.6


## 6. 三档对照

这就是 7.8 正文那张表的最小可运行版本：**回归把两种正确做法平均成一种撞墙的做法；
分类与扩散能表达"二选一"，但采样必须配合承诺（分块）才能执行**。
课程档把同样的对照搬到真 LeRobot 数据与真 ACT/扩散策略上，结论不变，只是任务从二维平面变成机械臂。

In [9]:
print(f"{'策略':<22}{'闭环成功率':>10}{'平均撞墙':>10}")
for name, result in [
    ('MSE 回归', mse_result),
    ('离散分类+承诺', cls_result),
    ('扩散+每步重规划', replan_result),
    ('扩散+分块执行', dif_result),
]:
    print(f"{name:<22}{result['success_rate']:>10.2f}{result['mean_collisions']:>10.1f}")

策略                         闭环成功率      平均撞墙
MSE 回归                      0.66       7.6
离散分类+承诺                     1.00       0.0
扩散+每步重规划                    0.44       2.6
扩散+分块执行                     0.91       0.0


## 7. 接到真 LeRobot（课程档入口）

冒烟档的数据是自造的。课程档要求换成真 `LeRobotDataset`：下面这段在有 `lerobot` 的环境里
提示如何导出标准格式；没有该环境会打印跳过提示，不影响冒烟结论。

In [10]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    print('lerobot 可用：按 7.8 第一步的字段表创建数据集，把 states/actions 写入 observation.state / action。')
except ImportError:
    print('未安装 lerobot，跳过导出。冒烟档的三档对照已经完成；课程档请先 pip install lerobot。')

未安装 lerobot，跳过导出。冒烟档的三档对照已经完成；课程档请先 pip install lerobot。
